
# DAZONE 2026 Round 3.1 — Multi-head PTONet + PUL-inspired Custom Loss

Notebook này triển khai kỹ thuật **Multi-head Neural Network + Custom Loss theo hướng PTONet/PUL** từ paper *Rethinking Causal Ranking: A Balanced Perspective on Uplift Model Evaluation*.

Mục tiêu của notebook:
- Huấn luyện **một mô hình đa nhiệm** học đồng thời 2 target: `cash_sign_30d` và `card_sign_30d`.
- Dự đoán uplift theo đúng bài toán DAZONE:
  - `score_cash = P(cash_sign_30d=1 | Cash) - P(cash_sign_30d=1 | Card)`
  - `score_card = P(card_sign_30d=1 | Card) - P(card_sign_30d=1 | Cash)`
- Đánh giá bằng **5-fold OOF Qini**.
- Luôn in 3 cột chính: `Qini_coefficient_cash`, `Qini_coefficient_card`, `Qini_coefficient_total`.

Ghi chú áp dụng paper:
- Paper gốc xử lý bài toán binary treatment + binary outcome.
- Bài DAZONE có binary treatment nhưng **2 outcome**, nên notebook mở rộng thành **shared trunk + 2 outcome heads**.
- Loss gồm BCE outcome, propensity/treatment reconstruction, targeted regularization, và **PUL-inspired contrast loss** cho từng target.


In [ ]:

# ============================================================
# 0. IMPORTS + GLOBAL CONFIG
# ============================================================
import gc
import os
import re
import time
import random
import warnings
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5

# Deep learning config: giữ tương đối nhẹ vì dữ liệu không quá lớn và tabular NN dễ overfit.
N_EPOCHS = 35
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 128
DROPOUT = 0.05
GRAD_CLIP_NORM = 3.0

# Loss weights.
LAMBDA_OUTCOME = 1.00
LAMBDA_PUL = 0.45
LAMBDA_TREAT = 0.10
LAMBDA_TARGETED = 0.10
TAU_LOGIT_SCALE = 4.0
EPS = 1e-6

# Debug nhanh: set DEBUG_SAMPLE_N = 5000 nếu muốn smoke test. Mặc định None để chạy full data.
DEBUG_SAMPLE_N = None

MODEL_NAME = "MULTIHEAD_PTONET_PUL"
TARGET_CASH = "cash_sign_30d"
TARGET_CARD = "card_sign_30d"
TREAT_COL_BINARY = "treatment_cash"
TREAT_COL_RAW = "treatment"
ID_CANDIDATES = ["id", "ID", "row_id", "customer_id", "Customer_ID", "client_id", "cli_id", "app_id", "application_id"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False

seed_everything(SEED)



## 1. Load dữ liệu và feature dropping tùy chọn

Notebook ưu tiên đọc cặp file đã clean:
- `df_feature.csv`: feature đã xử lý, có cột `treatment_cash`.
- `df_target.csv`: 2 target `cash_sign_30d`, `card_sign_30d`.

`COL_IDX` cho phép drop feature theo danh sách ưu tiên nếu cần thử nghiệm thêm. Mặc định `None` = không drop.


In [ ]:

# ============================================================
# 1. LOAD DATA + OPTIONAL FEATURE DROPPING
# ============================================================
FEATURE_DROP_PRIORITY_LIST = [
    "cnt_month_over3_credit_card_6m",
    "cnt_connected_tls",
    "cnt_earlyterminted_pos",
    "min_sign_vol",
    "cli_contact_region_population_grp",
    "cnt_inbound_tls_12m",
    "cnt_bank_inst_appl_12m",
    "f_approve_last_process",
    "worst_credit_status",
    "cli_contact_region_level",
    "cli_education",
    "f_ever_default",
    "cli_age",
    "cli_job",
    "cnt_earlyterminated_cash",
    "amt_terminated_loan_12m",
    "cnt_noinsurance_loan_24m",
    "cnt_report_bank_6m",
    "ratio_conn_tls_endmonth_12m",
    "ratio_conn_tls_weekend_12m",
    "cli_marital_status",
    "avg_sign_vol",
    "cnt_terminated_loan_12m",
    "cnt_fail_tls_3m",
    "last_offer_limit",
    "cli_emp_period",
    "cnt_fi_inst_appl_12m",
    "cnt_total_payment",
    "risk_grp",
    "cnt_lender_has2pay_last_24m",
    "cnt_month_status1",
    "last_process_channel",
    "cnt_month_from_last_appl",
    "cnt_report_institution_12m",
]

# None -> không drop.
# int -> drop 1 feature theo ID trong FEATURE_DROP_PRIORITY_LIST.
# list/range -> drop nhiều ID hoặc tên feature.
# string "a:b" -> drop ID từ a đến b-1.
COL_IDX = None


def find_file_pair(feature_name="df_feature.csv", target_name="df_target.csv"):
    roots = []
    for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "data", Path("/mnt/data")]:
        if p.exists() and str(p.resolve()) not in ["/", "/mnt"]:
            roots.append(p)

    candidates = []
    for root in roots:
        direct_feat = root / feature_name
        direct_tgt = root / target_name
        if direct_feat.exists() and direct_tgt.exists():
            return direct_feat, direct_tgt
        try:
            for feat_path in root.rglob(feature_name):
                tgt_path = feat_path.parent / target_name
                if tgt_path.exists():
                    candidates.append((feat_path, tgt_path))
        except Exception:
            pass

    if candidates:
        return sorted(candidates, key=lambda pair: len(str(pair[0])))[0]
    raise FileNotFoundError(
        f"Không tìm thấy {feature_name} và {target_name}. Hãy kiểm tra folder Kaggle/input hoặc working."
    )


def make_id_series(df, id_col=None):
    if id_col is not None and id_col in df.columns:
        return df[id_col].copy()
    return pd.Series(np.arange(len(df)), name="row_id")


def parse_col_idx(col_idx):
    if col_idx is None:
        return []
    if isinstance(col_idx, int):
        return [FEATURE_DROP_PRIORITY_LIST[col_idx]]
    if isinstance(col_idx, range):
        return [FEATURE_DROP_PRIORITY_LIST[i] for i in col_idx]
    if isinstance(col_idx, str):
        s = col_idx.strip()
        if re.fullmatch(r"\d+\s*:\s*\d+", s):
            a, b = [int(x.strip()) for x in s.split(":")]
            return [FEATURE_DROP_PRIORITY_LIST[i] for i in range(a, b)]
        if re.fullmatch(r"\d+", s):
            return [FEATURE_DROP_PRIORITY_LIST[int(s)]]
        return [s]
    if isinstance(col_idx, (list, tuple, set)):
        out = []
        for item in col_idx:
            if isinstance(item, int):
                out.append(FEATURE_DROP_PRIORITY_LIST[item])
            else:
                out.append(str(item))
        return out
    raise ValueError(f"COL_IDX không hợp lệ: {col_idx}")


def match_base_feature_columns(columns, base_feature):
    columns = list(columns)
    matched = []
    for c in columns:
        if c == base_feature:
            matched.append(c)
        elif c.startswith(base_feature + "_"):
            matched.append(c)
    return matched


feature_path, target_path = find_file_pair()
feat = pd.read_csv(feature_path)
target = pd.read_csv(target_path)

print("Feature path:", feature_path)
print("Target path :", target_path)
print("Raw feature shape:", feat.shape)
print("Raw target shape :", target.shape)

assert len(feat) == len(target), "df_feature và df_target khác số dòng."
assert TARGET_CASH in target.columns and TARGET_CARD in target.columns, "df_target thiếu target cash/card."
assert TREAT_COL_BINARY in feat.columns, "df_feature cần có cột treatment_cash."

ID_COL = next((col for col in ID_CANDIDATES if col in feat.columns), None)
row_id = make_id_series(feat, ID_COL)

Y_cash = target[TARGET_CASH].astype(int).to_numpy()
Y_card = target[TARGET_CARD].astype(int).to_numpy()
t_cash = feat[TREAT_COL_BINARY].astype(int).to_numpy()  # 1 = Cash, 0 = Card
w_cash = t_cash.copy()       # Event treatment của cash target = Cash
w_card = 1 - t_cash          # Event treatment của card target = Card

FORBIDDEN_FEATURE_COLS = {TARGET_CASH, TARGET_CARD, TREAT_COL_BINARY, TREAT_COL_RAW}
if ID_COL is not None:
    FORBIDDEN_FEATURE_COLS.add(ID_COL)

base_feature_cols = [c for c in feat.columns if c not in FORBIDDEN_FEATURE_COLS]

drop_bases = parse_col_idx(COL_IDX)
drop_cols = []
for base in drop_bases:
    drop_cols.extend(match_base_feature_columns(base_feature_cols, base))
drop_cols = sorted(set(drop_cols))

FEAT_COLS = [c for c in base_feature_cols if c not in drop_cols]

if DEBUG_SAMPLE_N is not None:
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(feat), size=min(DEBUG_SAMPLE_N, len(feat)), replace=False)
    idx = np.sort(idx)
    feat = feat.iloc[idx].reset_index(drop=True)
    target = target.iloc[idx].reset_index(drop=True)
    row_id = row_id.iloc[idx].reset_index(drop=True)
    Y_cash = Y_cash[idx]
    Y_card = Y_card[idx]
    t_cash = t_cash[idx]
    w_cash = w_cash[idx]
    w_card = w_card[idx]
    print(f"DEBUG_SAMPLE_N active -> using {len(idx)} rows")

X_df = feat[FEAT_COLS].copy()
X = X_df.to_numpy(dtype=np.float32)

print("Dropped base features:", drop_bases if drop_bases else "None")
print("Dropped columns:", drop_cols if drop_cols else "None")
print("Final X shape:", X.shape)
print("Treatment cash distribution:", pd.Series(t_cash).value_counts().sort_index().to_dict())
print("Target means:", {"cash": float(Y_cash.mean()), "card": float(Y_card.mean())})

# Leakage audit.
assert TARGET_CASH not in FEAT_COLS and TARGET_CARD not in FEAT_COLS, "Leakage: target nằm trong feature."
assert TREAT_COL_BINARY not in FEAT_COLS and TREAT_COL_RAW not in FEAT_COLS, "Leakage: treatment nằm trong feature."
if ID_COL is not None:
    assert ID_COL not in FEAT_COLS, "Risk/leakage: ID nằm trong feature."



## 2. Qini metric và fixed 5-fold

Cách tính giữ đúng logic đang dùng trong notebook `INT_RF_UPLIFT_RF_CIT.ipynb`:
- Cash: event treatment = Cash, control = Card.
- Card: event treatment = Card, control = Cash.


In [ ]:

# ============================================================
# 2. QINI METRIC + FIXED FOLDS
# ============================================================
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def qini_curve(y, w_event, score):
    y = np.asarray(y, dtype=float)
    w_event = np.asarray(w_event, dtype=int)
    score = np.asarray(score, dtype=float)

    order = np.argsort(-score, kind="mergesort")
    y_s = y[order]
    w_s = w_event[order]

    n_t = max(int(w_event.sum()), 1)
    n_c = max(len(w_event) - n_t, 1)
    q = np.cumsum(y_s * w_s) / n_t - np.cumsum(y_s * (1 - w_s)) / n_c
    x = np.arange(1, len(y) + 1) / len(y)
    return np.r_[0.0, x], np.r_[0.0, q]


def oracle_score(y, w_event):
    y = np.asarray(y, dtype=int)
    w_event = np.asarray(w_event, dtype=int)
    return w_event * y - (1 - w_event) * y


def qini_metrics(y, w_event, score):
    x, q = qini_curve(y, w_event, score)
    auc_model = float(_trapz(q, x))
    ate = float(q[-1])
    auc_random = ate * 0.5
    _, q_perfect = qini_curve(y, w_event, oracle_score(y, w_event))
    auc_perfect = float(_trapz(q_perfect, x))

    qini_diff = auc_model - auc_random
    denom = auc_perfect - auc_random
    qini_norm = qini_diff / denom if abs(denom) > 1e-12 else np.nan
    qini_ratio = auc_model / auc_random if abs(auc_random) > 1e-12 else np.nan
    return {
        "ate": ate,
        "auc_model": auc_model,
        "auc_random": auc_random,
        "auc_perfect": auc_perfect,
        "qini_diff": qini_diff,
        "qini_norm": qini_norm,
        "qini_ratio": qini_ratio,
    }


def make_multitask_fold_id(t_cash, y_cash, y_card):
    # Ưu tiên stratify theo tổ hợp treatment + 2 target để fold công bằng cho cả cash/card.
    key_full = (
        np.asarray(t_cash, dtype=int) * 4
        + np.asarray(y_cash, dtype=int) * 2
        + np.asarray(y_card, dtype=int)
    ).astype(int)
    vc_full = pd.Series(key_full).value_counts()

    dummy_x = np.zeros(len(key_full))
    fold_id = np.zeros(len(key_full), dtype=int)

    if vc_full.min() >= N_SPLITS:
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        split_iter = splitter.split(dummy_x, key_full)
        strat_desc = "treatment + cash target + card target"
    else:
        # Fallback cho DEBUG_SAMPLE_N nhỏ hoặc dữ liệu cực lệch: stratify theo treatment.
        key_t = np.asarray(t_cash, dtype=int)
        vc_t = pd.Series(key_t).value_counts()
        if vc_t.min() >= N_SPLITS:
            print(
                "Warning: một số stratum full ít hơn N_SPLITS, "
                "fallback sang stratify theo treatment_cash. Full strata:",
                vc_full.to_dict(),
            )
            splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
            split_iter = splitter.split(dummy_x, key_t)
            strat_desc = "treatment only"
        else:
            print(
                "Warning: treatment strata cũng ít hơn N_SPLITS, fallback sang KFold thường. "
                "Chỉ nên xảy ra khi DEBUG_SAMPLE_N quá nhỏ."
            )
            splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
            split_iter = splitter.split(dummy_x)
            strat_desc = "plain KFold"

    for fold, (_, va) in enumerate(split_iter, 1):
        fold_id[va] = fold
    print("Fold strategy:", strat_desc)
    return fold_id


fold_id = make_multitask_fold_id(t_cash, Y_cash, Y_card)
print("Fold distribution:", pd.Series(fold_id).value_counts().sort_index().to_dict())
print("Strata distribution:")
display(pd.Series(t_cash * 4 + Y_cash * 2 + Y_card).value_counts().sort_index().rename("count").to_frame())



## 3. Multi-head PTONet + PUL-inspired loss

Thiết kế:
- `x_net`: shared trunk học representation từ feature.
- `h_net`: nhận `[representation, treatment_cash]` để học outcome conditional theo treatment.
- `cash_head`, `card_head`: 2 head cho 2 target.
- `propensity_head`, `t_recon_head`: regularization theo PTONet.
- `PUL-inspired loss`: ép ranking uplift cân bằng hơn bằng target tương phản:
  - Cash target: `t_cash*y_cash + (1-t_cash)*(1-y_cash)`.
  - Card target: `t_card*y_card + (1-t_card)*(1-y_card)`.

Lưu ý: đây là phiên bản **adapted** cho DAZONE 2-outcome. Metric cuối vẫn là `Qini_total` theo đề bài.


In [ ]:

# ============================================================
# 3. DATASET + MODEL + LOSS
# ============================================================
class UpliftMultiTaskDataset(Dataset):
    def __init__(self, X, t_cash, y_cash, y_card):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.t_cash = torch.tensor(t_cash, dtype=torch.float32).view(-1, 1)
        self.y_cash = torch.tensor(y_cash, dtype=torch.float32).view(-1, 1)
        self.y_card = torch.tensor(y_card, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.t_cash[idx], self.y_cash[idx], self.y_card[idx]


class MultiHeadPTONetPUL(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.05):
        super().__init__()
        self.x_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout),
        )
        self.propensity_head = nn.Linear(hidden_dim, 1)

        self.h_net = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ELU(),
        )
        self.cash_head = nn.Linear(hidden_dim // 2, 1)
        self.card_head = nn.Linear(hidden_dim // 2, 1)
        self.t_recon_head = nn.Linear(hidden_dim // 2, 1)

        # Targeted regularization scalars, tương tự epsilon trong PTONet.
        self.eps_cash = nn.Parameter(torch.zeros(1))
        self.eps_card = nn.Parameter(torch.zeros(1))

    def forward(self, x, t_cash):
        if t_cash.dim() == 1:
            t_cash = t_cash.view(-1, 1)
        z = self.x_net(x)
        e_cash_logit = self.propensity_head(z)
        h = self.h_net(torch.cat([z, t_cash], dim=1))
        cash_logit = self.cash_head(h)
        card_logit = self.card_head(h)
        t_recon_logit = self.t_recon_head(h)
        return {
            "cash_logit": cash_logit,
            "card_logit": card_logit,
            "t_recon_logit": t_recon_logit,
            "e_cash_logit": e_cash_logit,
        }

    @torch.no_grad()
    def predict_uplift(self, X_np, batch_size=8192, device=DEVICE):
        self.eval()
        cash_scores, card_scores = [], []
        X_tensor = torch.tensor(X_np, dtype=torch.float32)
        for start in range(0, len(X_tensor), batch_size):
            xb = X_tensor[start:start + batch_size].to(device)
            t1 = torch.ones((len(xb), 1), dtype=torch.float32, device=device)   # Cash strategy
            t0 = torch.zeros((len(xb), 1), dtype=torch.float32, device=device)  # Card strategy
            out_cash = self.forward(xb, t1)
            out_card = self.forward(xb, t0)

            p_cash_if_cash = torch.sigmoid(out_cash["cash_logit"])
            p_cash_if_card = torch.sigmoid(out_card["cash_logit"])
            p_card_if_card = torch.sigmoid(out_card["card_logit"])
            p_card_if_cash = torch.sigmoid(out_cash["card_logit"])

            cash_scores.append((p_cash_if_cash - p_cash_if_card).detach().cpu().numpy().ravel())
            card_scores.append((p_card_if_card - p_card_if_cash).detach().cpu().numpy().ravel())
        return np.concatenate(cash_scores), np.concatenate(card_scores)


def get_pos_weight(y):
    y = np.asarray(y, dtype=int)
    pos = max(int(y.sum()), 1)
    neg = max(len(y) - pos, 1)
    return torch.tensor([neg / pos], dtype=torch.float32, device=DEVICE)


def weighted_bce_logits(logit, target, pos_weight):
    return F.binary_cross_entropy_with_logits(logit, target, pos_weight=pos_weight)


def compute_multitask_ptonet_loss(model, batch, pos_weight_cash, pos_weight_card):
    x, t_cash_b, y_cash_b, y_card_b = batch
    x = x.to(DEVICE)
    t_cash_b = t_cash_b.to(DEVICE)
    y_cash_b = y_cash_b.to(DEVICE)
    y_card_b = y_card_b.to(DEVICE)
    t_card_b = 1.0 - t_cash_b

    # Observed factual predictions.
    out_obs = model(x, t_cash_b)
    cash_logit_obs = out_obs["cash_logit"]
    card_logit_obs = out_obs["card_logit"]
    e_cash_logit = out_obs["e_cash_logit"]
    t_recon_logit = out_obs["t_recon_logit"]

    outcome_loss = (
        weighted_bce_logits(cash_logit_obs, y_cash_b, pos_weight_cash)
        + weighted_bce_logits(card_logit_obs, y_card_b, pos_weight_card)
    ) / 2.0

    # Treatment/propensity regularization.
    treat_loss = (
        F.binary_cross_entropy_with_logits(t_recon_logit, t_cash_b)
        + F.binary_cross_entropy_with_logits(e_cash_logit, t_cash_b)
    ) / 2.0

    # Targeted regularization.
    e_cash = torch.sigmoid(e_cash_logit).clamp(EPS, 1.0 - EPS)
    h_cash = t_cash_b / e_cash - (1.0 - t_cash_b) / (1.0 - e_cash)
    h_card = t_card_b / (1.0 - e_cash) - (1.0 - t_card_b) / e_cash

    cash_perturbed = cash_logit_obs + model.eps_cash * h_cash
    card_perturbed = card_logit_obs + model.eps_card * h_card
    targeted_loss = (
        weighted_bce_logits(cash_perturbed, y_cash_b, pos_weight_cash)
        + weighted_bce_logits(card_perturbed, y_card_b, pos_weight_card)
    ) / 2.0

    # Counterfactual-style potential predictions for PUL-inspired contrast.
    t1 = torch.ones_like(t_cash_b)   # Cash
    t0 = torch.zeros_like(t_cash_b)  # Card
    out_t1 = model(x, t1)
    out_t0 = model(x, t0)

    p_cash_t1 = torch.sigmoid(out_t1["cash_logit"])
    p_cash_t0 = torch.sigmoid(out_t0["cash_logit"])
    p_card_t1 = torch.sigmoid(out_t1["card_logit"])
    p_card_t0 = torch.sigmoid(out_t0["card_logit"])

    tau_cash = p_cash_t1 - p_cash_t0       # Cash uplift: Cash - Card
    tau_card = p_card_t0 - p_card_t1       # Card uplift: Card - Cash

    # PUL-inspired target: high uplift cho treated positive và control negative.
    pul_target_cash = t_cash_b * y_cash_b + (1.0 - t_cash_b) * (1.0 - y_cash_b)
    pul_target_card = t_card_b * y_card_b + (1.0 - t_card_b) * (1.0 - y_card_b)

    pul_loss = (
        F.binary_cross_entropy_with_logits(TAU_LOGIT_SCALE * tau_cash, pul_target_cash)
        + F.binary_cross_entropy_with_logits(TAU_LOGIT_SCALE * tau_card, pul_target_card)
    ) / 2.0

    total_loss = (
        LAMBDA_OUTCOME * outcome_loss
        + LAMBDA_PUL * pul_loss
        + LAMBDA_TREAT * treat_loss
        + LAMBDA_TARGETED * targeted_loss
    )

    return total_loss, {
        "loss": float(total_loss.detach().cpu()),
        "outcome": float(outcome_loss.detach().cpu()),
        "pul": float(pul_loss.detach().cpu()),
        "treat": float(treat_loss.detach().cpu()),
        "targeted": float(targeted_loss.detach().cpu()),
    }



## 4. Train 5-fold và in Qini cash/card/total


In [ ]:

# ============================================================
# 4. TRAIN 5-FOLD OOF
# ============================================================
def train_one_fold(fold, tr_idx, va_idx):
    seed_everything(SEED + fold)

    model = MultiHeadPTONetPUL(
        input_dim=X.shape[1],
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)

    train_ds = UpliftMultiTaskDataset(X[tr_idx], t_cash[tr_idx], Y_cash[tr_idx], Y_card[tr_idx])
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
    )

    pos_weight_cash = get_pos_weight(Y_cash[tr_idx])
    pos_weight_card = get_pos_weight(Y_card[tr_idx])

    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)

    history = []
    start = time.time()
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        epoch_logs = []
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            loss, log_items = compute_multitask_ptonet_loss(model, batch, pos_weight_cash, pos_weight_card)
            loss.backward()
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            opt.step()
            epoch_logs.append(log_items)
        scheduler.step()

        mean_log = {k: float(np.mean([d[k] for d in epoch_logs])) for k in epoch_logs[0].keys()}
        mean_log["epoch"] = epoch
        history.append(mean_log)

        # In gọn để tránh log quá dài.
        if epoch in [1, 2, 3, N_EPOCHS] or epoch % 10 == 0:
            print(
                f"    epoch {epoch:02d}/{N_EPOCHS} | "
                f"loss={mean_log['loss']:.5f} | outcome={mean_log['outcome']:.5f} | "
                f"pul={mean_log['pul']:.5f} | targeted={mean_log['targeted']:.5f}"
            )

    cash_pred_va, card_pred_va = model.predict_uplift(X[va_idx], device=DEVICE)

    cash_met = qini_metrics(Y_cash[va_idx], w_cash[va_idx], cash_pred_va)
    card_met = qini_metrics(Y_card[va_idx], w_card[va_idx], card_pred_va)
    total_qini = cash_met["qini_norm"] + card_met["qini_norm"]

    print(
        f"  fold {fold}: "
        f"Qini_coefficient_cash={cash_met['qini_norm']:+.5f} | "
        f"Qini_coefficient_card={card_met['qini_norm']:+.5f} | "
        f"Qini_coefficient_total={total_qini:+.5f} | "
        f"minutes={(time.time() - start) / 60:.2f}"
    )

    return model, cash_pred_va, card_pred_va, pd.DataFrame(history)


print(f"\n[{MODEL_NAME}] 5-fold training")
oof_cash = np.zeros(len(X), dtype=np.float32)
oof_card = np.zeros(len(X), dtype=np.float32)
fold_logs = []
history_by_fold = {}
fold_models = []

start_all = time.time()
for fold in range(1, N_SPLITS + 1):
    print(f"\n[{MODEL_NAME}] Fold {fold}/{N_SPLITS}")
    va_idx = np.where(fold_id == fold)[0]
    tr_idx = np.where(fold_id != fold)[0]

    model, cash_pred_va, card_pred_va, hist = train_one_fold(fold, tr_idx, va_idx)
    oof_cash[va_idx] = cash_pred_va
    oof_card[va_idx] = card_pred_va
    history_by_fold[fold] = hist
    fold_models.append(model)

    cash_met = qini_metrics(Y_cash[va_idx], w_cash[va_idx], cash_pred_va)
    card_met = qini_metrics(Y_card[va_idx], w_card[va_idx], card_pred_va)
    fold_logs.append({
        "model": MODEL_NAME,
        "fold": fold,
        "Qini_coefficient_cash": cash_met["qini_norm"],
        "Qini_coefficient_card": card_met["qini_norm"],
        "Qini_coefficient_total": cash_met["qini_norm"] + card_met["qini_norm"],
        "Q_ratio_cash": cash_met["qini_ratio"],
        "Q_ratio_card": card_met["qini_ratio"],
    })
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"\nTotal runtime: {(time.time() - start_all) / 60:.2f} minutes")


In [ ]:

# ============================================================
# 5. SUMMARY TABLES
# ============================================================
fold_qini_logs = pd.DataFrame(fold_logs)

cash_oof_met = qini_metrics(Y_cash, w_cash, oof_cash)
card_oof_met = qini_metrics(Y_card, w_card, oof_card)

summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "Qini_coefficient_cash": cash_oof_met["qini_norm"],
    "Qini_coefficient_card": card_oof_met["qini_norm"],
    "Qini_coefficient_total": cash_oof_met["qini_norm"] + card_oof_met["qini_norm"],
    "Q_ratio_cash": cash_oof_met["qini_ratio"],
    "Q_ratio_card": card_oof_met["qini_ratio"],
}])

fold_mean = (
    fold_qini_logs
    .groupby("model", as_index=False)
    .agg(
        n_folds=("fold", "nunique"),
        Mean_Qini_coefficient_cash=("Qini_coefficient_cash", "mean"),
        Mean_Qini_coefficient_card=("Qini_coefficient_card", "mean"),
        Mean_Qini_coefficient_total=("Qini_coefficient_total", "mean"),
    )
)

print("OOF Qini coefficient summary — CASH / CARD / TOTAL")
display(summary[["model", "Qini_coefficient_cash", "Qini_coefficient_card", "Qini_coefficient_total", "Q_ratio_cash", "Q_ratio_card"]].round(6))

print("Fold Qini logs")
display(fold_qini_logs[["model", "fold", "Qini_coefficient_cash", "Qini_coefficient_card", "Qini_coefficient_total"]].round(6))

print("Fold mean Qini by model")
display(fold_mean.round(6))



## 5. Qini curves


In [ ]:

# ============================================================
# 6. QINI CURVES
# ============================================================
def plot_qini_single(title, y, w_event, score):
    plt.figure(figsize=(9, 5))

    x, q = qini_curve(y, w_event, score)
    met = qini_metrics(y, w_event, score)

    xp, qp = qini_curve(y, w_event, oracle_score(y, w_event))
    ate = q[-1]

    plt.plot([0, 1], [0, ate], "--", linewidth=1.2, label=f"Random baseline | ATE={ate:+.5f}")
    plt.plot(xp, qp, ":", linewidth=1.4, label="Perfect ranking")
    plt.plot(x, q, linewidth=2.0, label=f"{MODEL_NAME} | Qini={met['qini_norm']:+.4f}")

    plt.title(title)
    plt.xlabel("Fraction of population sorted by predicted uplift")
    plt.ylabel("Qini")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_qini_single("Qini Curve — CASH", Y_cash, w_cash, oof_cash)
plot_qini_single("Qini Curve — CARD", Y_card, w_card, oof_card)



## 6. Optional test prediction / submission

Nếu có `df_feature_test.csv` hoặc `test_feature.csv` đã one-hot/scale giống train, notebook sẽ dùng trực tiếp.

Nếu chỉ có raw `dazone_sample_test.csv`, notebook có fallback preprocess lại raw test bằng rule suy ra từ train raw/cleaned:
- numeric: median imputation + min-max scale theo train.
- categorical: mode imputation + one-hot `drop_first` theo train.
- `cli_education`: ordinal mapping `ELEMENTARY_SCHOOL=0`, `PTCS=0.25`, `PTTH=0.5`, `BACHELORS=0.75`, `MASTERS=1.0`.

Output gồm: `score_cash`, `score_card`, `recommended_treatment`.


In [ ]:

# ============================================================
# 7. OPTIONAL TEST PREDICTION
# ============================================================
def find_first_existing_file(names):
    roots = []
    for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "data", Path("/mnt/data")]:
        if p.exists() and str(p.resolve()) not in ["/", "/mnt"]:
            roots.append(p)
    for root in roots:
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
            try:
                hits = list(root.rglob(name))
                if hits:
                    return sorted(hits, key=lambda p: len(str(p)))[0]
            except Exception:
                pass
    return None


def build_test_matrix_from_clean_feature(test_feature_path):
    test_feat = pd.read_csv(test_feature_path)
    test_id_col = next((col for col in ID_CANDIDATES if col in test_feat.columns), None)
    test_id = make_id_series(test_feat, test_id_col)
    X_test_df = test_feat.reindex(columns=FEAT_COLS, fill_value=0).copy()
    return test_id, X_test_df.to_numpy(dtype=np.float32)


def fit_raw_preprocessor(train_raw):
    # Khớp với cách tạo df_feature.csv: numeric median + minmax, categorical mode + one-hot drop_first.
    cat_dummy_cols = [
        "cli_job",
        "cli_marital_status",
        "cli_contact_region_level",
        "last_process_channel",
        "risk_grp",
        "cli_contact_region_population_grp",
    ]
    edu_col = "cli_education"
    target_cols = [TARGET_CASH, TARGET_CARD]
    id_cols = ["customer_id", "Unnamed: 0", "index"]
    exclude = set(target_cols + id_cols + [TREAT_COL_RAW, edu_col] + cat_dummy_cols)
    num_cols = [c for c in train_raw.columns if c not in exclude]

    num_train = train_raw[num_cols].apply(pd.to_numeric, errors="coerce")
    med = num_train.median(numeric_only=True)
    num_filled = num_train.fillna(med)
    mins = num_filled.min()
    maxs = num_filled.max()
    denoms = (maxs - mins).replace(0, 1.0)

    cat_info = {}
    for c in cat_dummy_cols:
        mode = train_raw[c].mode(dropna=True).iloc[0]
        s = train_raw[c].fillna(mode).astype(str)
        cats = sorted(s.unique())
        cat_info[c] = {"mode": mode, "categories": cats, "dummy_values": cats[1:]}

    edu_map = {
        "ELEMENTARY_SCHOOL": 0.0,
        "PTCS": 0.25,
        "PTTH": 0.50,
        "BACHELORS": 0.75,
        "MASTERS": 1.00,
    }
    return {
        "num_cols": num_cols,
        "med": med,
        "mins": mins,
        "denoms": denoms,
        "cat_info": cat_info,
        "edu_map": edu_map,
    }


def transform_raw_with_preprocessor(raw_df, prep, final_columns):
    out = pd.DataFrame(index=raw_df.index)

    for c in prep["num_cols"]:
        if c in raw_df.columns:
            vals = pd.to_numeric(raw_df[c], errors="coerce").fillna(prep["med"][c])
        else:
            vals = pd.Series(prep["med"][c], index=raw_df.index)
        out[c] = ((vals - prep["mins"][c]) / prep["denoms"][c]).astype(float)

    if "cli_education" in raw_df.columns:
        out["cli_education"] = raw_df["cli_education"].map(prep["edu_map"]).fillna(0.50).astype(float)
    else:
        out["cli_education"] = 0.50

    for c, info in prep["cat_info"].items():
        if c in raw_df.columns:
            s = raw_df[c].fillna(info["mode"]).astype(str)
        else:
            s = pd.Series(info["mode"], index=raw_df.index, dtype=str)
        for val in info["dummy_values"]:
            out[f"{c}_{val}"] = (s == val).astype(int)

    # treatment_cash có thể có trong df_feature train nhưng không được dùng trong X.
    if TREAT_COL_RAW in raw_df.columns:
        out[TREAT_COL_BINARY] = (raw_df[TREAT_COL_RAW].astype(str).str.lower() == "cash").astype(int)

    return out.reindex(columns=final_columns, fill_value=0).astype(np.float32)


def build_test_matrix_from_raw(raw_test_path):
    train_raw_path = find_first_existing_file(["cleaned_dazone_sample.csv", "dazone_sample.csv"])
    if train_raw_path is None:
        raise FileNotFoundError("Không tìm thấy cleaned_dazone_sample.csv/dazone_sample.csv để fit raw preprocessor.")

    train_raw = pd.read_csv(train_raw_path)
    test_raw = pd.read_csv(raw_test_path)
    test_id_col = next((col for col in ID_CANDIDATES if col in test_raw.columns), None)
    test_id = make_id_series(test_raw, test_id_col)

    prep = fit_raw_preprocessor(train_raw)
    X_test_df = transform_raw_with_preprocessor(test_raw, prep, FEAT_COLS)
    return test_id, X_test_df.to_numpy(dtype=np.float32)


clean_test_path = find_first_existing_file(["df_feature_test.csv", "test_feature.csv", "X_test.csv"])
raw_test_path = find_first_existing_file(["dazone_sample_test.csv"])

X_test = None
test_id = None

try:
    if clean_test_path is not None:
        print("Using clean test feature:", clean_test_path)
        test_id, X_test = build_test_matrix_from_clean_feature(clean_test_path)
    elif raw_test_path is not None:
        print("Using raw test fallback:", raw_test_path)
        test_id, X_test = build_test_matrix_from_raw(raw_test_path)
    else:
        print("Không tìm thấy test file -> skip submission.")
except Exception as e:
    print("Không build được test matrix -> skip submission. Error:", repr(e))
    X_test = None
    test_id = None

if X_test is not None:
    print("X_test shape:", X_test.shape)
    test_cash_scores = np.zeros(len(X_test), dtype=np.float32)
    test_card_scores = np.zeros(len(X_test), dtype=np.float32)

    for model in fold_models:
        cash_s, card_s = model.predict_uplift(X_test, device=DEVICE)
        test_cash_scores += cash_s.astype(np.float32) / len(fold_models)
        test_card_scores += card_s.astype(np.float32) / len(fold_models)

    recommended = np.where(test_cash_scores >= test_card_scores, "cash", "card")
    submission = pd.DataFrame({
        "customer_id": test_id.values,
        "score_cash": test_cash_scores,
        "score_card": test_card_scores,
        "recommended_treatment": recommended,
    })
    print("Submission preview:")
    display(submission.head())
else:
    submission = None



## 7. Export outputs


In [ ]:

# ============================================================
# 8. EXPORT OUTPUTS
# ============================================================
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUT_DIR / "summary_qini_multitask_ptonet_pul.csv"
fold_path = OUT_DIR / "fold_qini_multitask_ptonet_pul.csv"
oof_path = OUT_DIR / "oof_multitask_ptonet_pul.csv"
hist_path = OUT_DIR / "train_history_multitask_ptonet_pul.csv"

summary.to_csv(summary_path, index=False)
fold_qini_logs.to_csv(fold_path, index=False)

pd.DataFrame({
    "row_id": row_id.values,
    "treatment_cash": t_cash,
    "cash_sign_30d": Y_cash,
    "card_sign_30d": Y_card,
    "oof_score_cash": oof_cash,
    "oof_score_card": oof_card,
    "fold": fold_id,
}).to_csv(oof_path, index=False)

hist_all = []
for f, h in history_by_fold.items():
    hh = h.copy()
    hh.insert(0, "fold", f)
    hist_all.append(hh)
hist_all = pd.concat(hist_all, ignore_index=True) if hist_all else pd.DataFrame()
hist_all.to_csv(hist_path, index=False)

print("Saved:")
print("-", summary_path)
print("-", fold_path)
print("-", oof_path)
print("-", hist_path)

if submission is not None:
    sub_path = OUT_DIR / "submission_multitask_ptonet_pul.csv"
    submission.to_csv(sub_path, index=False)
    print("-", sub_path)
